<a href="https://colab.research.google.com/github/Sabari19-adda/A-Selective-Feature-Layer-and-Soft-Label-Fused-Knowledge-Distillation-System-/blob/main/DenseNet121_25_50_75epoch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# PART 1 — FULL SETUP (NEW COLAB NOTEBOOK READY)
# =========================================================

# -------------------------------
# 1. KAGGLE + DATASET DOWNLOAD
# -------------------------------
!pip install -q kaggle

from google.colab import files
files.upload()   # Upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset-v2
!unzip -q augmented-alzheimer-mri-dataset-v2.zip -d data

from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# 2. IMPORTS
# -------------------------------
import os, shutil, numpy as np, tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, SeparableConv2D, BatchNormalization,
                                     MaxPooling2D, GlobalAveragePooling2D,
                                     Dense, Dropout)
from tensorflow.keras.optimizers import Adam

# -------------------------------
# 3. MERGE TRAIN + VAL (SAME AS YOUR KD PIPELINE)
# -------------------------------
base_dir = 'data/data'
combined_dir = 'combined_data'
os.makedirs(combined_dir + '/all', exist_ok=True)

classes = os.listdir(os.path.join(base_dir, 'train'))

for cls in classes:
    os.makedirs(f"{combined_dir}/all/{cls}", exist_ok=True)

    for split in ['train','val']:
        src = os.path.join(base_dir, split, cls)
        if not os.path.exists(src):
            continue

        for img in os.listdir(src):
            shutil.copy(os.path.join(src,img),
                        f"{combined_dir}/all/{cls}")

# -------------------------------
# 4. SETTINGS
# -------------------------------
img_size = (224,224)
batch_size = 32
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

# -------------------------------
# 5. TRAIN / VAL GENERATORS (REQUIRED)
# -------------------------------
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=seed
)

val_gen = datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=seed
)

# -------------------------------
# 6. FILELIST GENERATOR (USED FOR KD SOFT LABELS)
# -------------------------------
filelist_datagen = ImageDataGenerator(rescale=1./255)

filelist_gen = filelist_datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

filepaths = filelist_gen.filepaths

labels_onehot = tf.keras.utils.to_categorical(
    filelist_gen.labels,
    num_classes=len(filelist_gen.class_indices)
)

print("Total images:", len(filepaths))

# -------------------------------
# 7. STUDENT MODEL (UNCHANGED)
# -------------------------------
def build_separable_model(input_shape=(224,224,3), num_classes=4):

    inputs = Input(shape=input_shape)

    x = SeparableConv2D(32,3,padding='same',activation='relu')(inputs)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(64,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(128,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(256,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(256,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(512,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = SeparableConv2D(512,3,padding='same',activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)

    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256,activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(filelist_gen.class_indices),activation='softmax')(x)

    model = Model(inputs,outputs,name="Student_SeparableConv")
    model.compile(optimizer=Adam(1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

student = build_separable_model()

print("✅ PART 1 COMPLETE — DATA + GENERATORS + STUDENT READY")
student.summary()


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset-v2
License(s): GNU Lesser General Public License 3.0
 79% 298M/379M [00:00<00:00, 870MB/s] 
100% 379M/379M [00:00<00:00, 630MB/s]
Mounted at /content/drive
Found 32308 images belonging to 4 classes.
Found 8076 images belonging to 4 classes.
Found 40384 images belonging to 4 classes.
Total images: 40384
✅ PART 1 COMPLETE — DATA + GENERATORS + STUDENT READY


Model: "Student_SeparableConv"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 224, 224, 32)   │           155 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 112, 112, 64)   │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 56, 56, 128)    │         8,896 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_3              │ (None, 28, 28, 256)    │        34,176 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_4              │ (None, 14, 14, 256)    │        68,096 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_5              │ (None, 7, 7, 512)      │       133,888 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 7, 7, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 3, 3, 512)      │             

 Total params: 655,295 (2.50 MB)

 Trainable params: 651,263 (2.48 MB)

 Non-trainable params: 4,032 (15.75 KB)

In [ ]:
# =========================================================
# PART 2 — STABILIZED FEATURE KD (DENSENET121 VERSION)
# =========================================================

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
import numpy as np
import tensorflow as tf

# -------------------------------
# LOAD DENSENET121 TEACHER
# -------------------------------
teacher_path = "/content/drive/MyDrive/Alzheimer_Models/DenseNet121_best_model.h5"
teacher = load_model(teacher_path)
teacher.trainable = False
print("✅ DenseNet121 Teacher loaded")

# -------------------------------
# GENERATE TEACHER SOFT LABELS
# -------------------------------
print("🔹 Generating teacher soft labels...")
teacher_probs = teacher.predict(filelist_gen, verbose=1)

# -------------------------------
# SPLIT DATA (80/20)
# -------------------------------
num_samples = len(filepaths)
indices = np.arange(num_samples)
np.random.shuffle(indices)

split = int(num_samples * 0.8)
train_idx, val_idx = indices[:split], indices[split:]

train_files = [filepaths[i] for i in train_idx]
val_files   = [filepaths[i] for i in val_idx]

train_labels = labels_onehot[train_idx]
val_labels   = labels_onehot[val_idx]

train_teacher = teacher_probs[train_idx]
val_teacher   = teacher_probs[val_idx]

# -------------------------------
# KDSequence (UNCHANGED)
# -------------------------------
class KDSequence(Sequence):
    def __init__(self, files, labels, teacher_probs):
        self.files = np.array(files)
        self.labels = np.array(labels)
        self.teacher_probs = np.array(teacher_probs)

    def __len__(self):
        return int(np.ceil(len(self.files) / batch_size))

    def __getitem__(self, idx):
        bf = self.files[idx*batch_size:(idx+1)*batch_size]
        bl = self.labels[idx*batch_size:(idx+1)*batch_size]
        bt = self.teacher_probs[idx*batch_size:(idx+1)*batch_size]

        imgs = np.zeros((len(bf),224,224,3))
        for i,p in enumerate(bf):
            imgs[i] = img_to_array(load_img(p,target_size=img_size))/255.0

        return (imgs.astype(np.float32), bt.astype(np.float32)), bl.astype(np.float32)

train_seq = KDSequence(train_files, train_labels, train_teacher)
val_seq   = KDSequence(val_files, val_labels, val_teacher)

# -------------------------------
# DENSENET121 FEATURE MODELS
# -------------------------------

# 🔥 Correct DenseNet121 feature layers
teacher_feat_layers = [
    "pool4_conv",   # mid-level dense block output
    "relu"          # final activation before classifier
]

student_feat_layers = [
    "separable_conv2d_3",
    "separable_conv2d_6"
]

teacher_feat_models = [
    Model(teacher.input, teacher.get_layer(n).output)
    for n in teacher_feat_layers
]

student_feat_models = [
    Model(student.input, student.get_layer(n).output)
    for n in student_feat_layers
]

gap = GlobalAveragePooling2D()

# -------------------------------
# STABILIZED FEATURE KD DISTILLER
# -------------------------------
class StableFeatureDistiller(tf.keras.Model):

    def __init__(self, student, teacher, t_feats, s_feats,
                 alpha=0.5, T=3.0, beta_max=0.25):
        super().__init__()

        self.student = student
        self.teacher = teacher
        self.t_feats = t_feats
        self.s_feats = s_feats
        self.alpha = alpha
        self.T = T
        self.beta_max = beta_max

        self.ce = CategoricalCrossentropy(from_logits=False)
        self.kld = KLDivergence()
        self.mse = MeanSquaredError()
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')

        # Projection heads match DenseNet feature size
        self.proj_heads = [
            Dense(tm.output_shape[-1], use_bias=False, name=f"proj_head_{i}")
            for i, tm in enumerate(self.t_feats)
        ]

    @property
    def metrics(self):
        return [self.acc_metric]

    def normalize_feat(self, f):
        return tf.math.l2_normalize(f, axis=1)

    def train_step(self, data):

        (x, t_probs), y = data

        step = tf.cast(self.optimizer.iterations, tf.float32)
        beta = tf.minimum(self.beta_max, step/5000.0*self.beta_max)

        with tf.GradientTape() as tape:

            s_logits = self.student(x, training=True)

            s_loss = self.ce(y, s_logits)

            kd_loss = self.kld(
                tf.nn.softmax(t_probs/self.T),
                tf.nn.softmax(s_logits/self.T)
            )*(self.T*self.T)

            feat_loss = 0.0

            for i,(tm,sm) in enumerate(zip(self.t_feats,self.s_feats)):

                t_feat = gap(tf.stop_gradient(tm(x, training=False)))
                s_feat = gap(sm(x, training=True))

                s_feat = self.proj_heads[i](s_feat)

                t_feat = self.normalize_feat(t_feat)
                s_feat = self.normalize_feat(s_feat)

                feat_loss += self.mse(t_feat, s_feat)

            loss = self.alpha*s_loss + (1-self.alpha)*kd_loss + beta*feat_loss

        train_vars = self.student.trainable_variables
        for ph in self.proj_heads:
            train_vars += ph.trainable_variables

        grads = tape.gradient(loss, train_vars)
        self.optimizer.apply_gradients(zip(grads, train_vars))

        self.acc_metric.update_state(y, s_logits)

        return {"loss":loss,"accuracy":self.acc_metric.result(),"beta":beta}

    def test_step(self,data):
        (x,t_probs),y=data
        s_logits=self.student(x,training=False)
        self.acc_metric.update_state(y,s_logits)
        return {"accuracy":self.acc_metric.result()}

# -------------------------------
# TRAIN DISTILLER
# -------------------------------
distiller = StableFeatureDistiller(
    student,
    teacher,
    teacher_feat_models,
    student_feat_models
)

distiller.compile(optimizer=Adam(1e-4))

best_student_path="/content/drive/MyDrive/best_student_densenet_stable_featureKD.keras"

class SaveBestStudent(tf.keras.callbacks.Callback):

    def __init__(self,path):
        super().__init__()
        self.best=-1.0
        self.path=path

    def on_epoch_end(self,epoch,logs=None):

        if logs is None:
            return

        val_acc=logs.get("val_accuracy")

        if val_acc is None:
            print("⚠️ val_accuracy not available — skipping save")
            return

        if val_acc>self.best:
            self.best=val_acc
            print(f"🔥 Saving Best Student Only (val_accuracy={val_acc:.4f})")
            self.model.student.save(self.path)

distiller.fit(
    train_seq,
    validation_data=val_seq,
    epochs=25,
    callbacks=[SaveBestStudent(best_student_path)]
)

distiller.student.save("/content/drive/MyDrive/final_student_densenet_stable_featureKD.keras")

print("✅ DENSENET121 FEATURE KD TRAINING COMPLETE")


✅ DenseNet121 Teacher loaded
🔹 Generating teacher soft labels...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1262/1262 ━━━━━━━━━━━━━━━━━━━━ 119s 80ms/step
Epoch 1/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step - accuracy: 0.3630 - beta: 0.0252 - loss: 0.8982🔥 Saving Best Student Only (val_accuracy=0.5964)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 328s 270ms/step - accuracy: 0.3630 - beta: 0.0252 - loss: 0.8979 - val_accuracy: 0.5964
Epoch 2/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.5056 - beta: 0.0757 - loss: 0.6337🔥 Saving Best Student Only (val_accuracy=0.6639)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 225s 222ms/step - accuracy: 0.5056 - beta: 0.0757 - loss: 0.6335 - val_accuracy: 0.6639
Epoch 3/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.5884 - beta: 0.1262 - loss: 0.4977🔥 Saving Best Student Only (val_accuracy=0.7186)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 226s 224ms/step - accuracy: 0.5884 - beta: 0.1262 - loss: 0.4976 - val_accuracy: 0.7186
Epoch 4/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.6643 - beta: 0.1767 - loss: 0.3972🔥 Saving Best Student Only (val_ac

In [ ]:
# =========================================================
# RUN 2 — CONTINUE TRAINING (RESNET101V2 STABILIZED FEATURE KD)
# =========================================================

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence, MeanSquaredError
from tensorflow.keras.optimizers import Adam
import os

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence, MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
import numpy as np
import tensorflow as tf


print("\n🚀 Starting RUN 2 — Loading BEST STUDENT from Run1")
# -------------------------------
# GENERATE TEACHER SOFT LABELS
# -------------------------------
print("🔹 Generating teacher soft labels...")
teacher_probs = teacher.predict(filelist_gen, verbose=1)

# -------------------------------
# SPLIT DATA (80/20)
# -------------------------------
num_samples = len(filepaths)
indices = np.arange(num_samples)
np.random.shuffle(indices)

split = int(num_samples * 0.8)
train_idx, val_idx = indices[:split], indices[split:]

train_files = [filepaths[i] for i in train_idx]
val_files   = [filepaths[i] for i in val_idx]

train_labels = labels_onehot[train_idx]
val_labels   = labels_onehot[val_idx]

train_teacher = teacher_probs[train_idx]
val_teacher   = teacher_probs[val_idx]

# -------------------------------
# KDSequence (UNCHANGED)
# -------------------------------
class KDSequence(Sequence):
    def __init__(self, files, labels, teacher_probs):
        self.files = np.array(files)
        self.labels = np.array(labels)
        self.teacher_probs = np.array(teacher_probs)

    def __len__(self):
        return int(np.ceil(len(self.files) / batch_size))

    def __getitem__(self, idx):
        bf = self.files[idx*batch_size:(idx+1)*batch_size]
        bl = self.labels[idx*batch_size:(idx+1)*batch_size]
        bt = self.teacher_probs[idx*batch_size:(idx+1)*batch_size]

        imgs = np.zeros((len(bf),224,224,3))
        for i,p in enumerate(bf):
            imgs[i] = img_to_array(load_img(p,target_size=img_size))/255.0

        return (imgs.astype(np.float32), bt.astype(np.float32)), bl.astype(np.float32)

train_seq = KDSequence(train_files, train_labels, train_teacher)
val_seq   = KDSequence(val_files, val_labels, val_teacher)

# ---------------------------------------------------------
# LOAD STUDENT FROM RUN1
# ---------------------------------------------------------
run1_best_student_path = "/content/drive/MyDrive/final_student_densenet_stable_featureKD.keras"

student = load_model(run1_best_student_path)

print("✅ Run1 student loaded successfully")

# ---------------------------------------------------------
# LOAD TEACHER AGAIN (SAFE PRACTICE)
# ---------------------------------------------------------
teacher_path = "/content/drive/MyDrive/Alzheimer_Models/DenseNet121_best_model.h5"

teacher = load_model(teacher_path)
teacher.trainable = False

print("✅ Teacher reloaded")

# -------------------------------
# DENSENET121 FEATURE MODELS
# -------------------------------

# 🔥 Correct DenseNet121 feature layers
teacher_feat_layers = [
    "pool4_conv",   # mid-level dense block output
    "relu"          # final activation before classifier
]

student_feat_layers = [
    "separable_conv2d_3",
    "separable_conv2d_6"
]

teacher_feat_models = [
    Model(teacher.input, teacher.get_layer(n).output)
    for n in teacher_feat_layers
]

student_feat_models = [
    Model(student.input, student.get_layer(n).output)
    for n in student_feat_layers
]

gap = GlobalAveragePooling2D()

# ---------------------------------------------------------
# STABILIZED DISTILLER (IDENTICAL LOGIC)
# ---------------------------------------------------------
class StableFeatureDistiller(tf.keras.Model):

    def __init__(self, student, teacher, t_feats, s_feats,
                 alpha=0.5, T=3.0, beta_max=0.25):
        super().__init__()

        self.student = student
        self.teacher = teacher
        self.t_feats = t_feats
        self.s_feats = s_feats
        self.alpha = alpha
        self.T = T
        self.beta_max = beta_max

        self.ce = CategoricalCrossentropy(from_logits=False)
        self.kld = KLDivergence()
        self.mse = MeanSquaredError()

        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')

        self.proj_heads = [
            Dense(tm.output_shape[-1], use_bias=False, name=f"proj_head_{i}")
            for i, tm in enumerate(self.t_feats)
        ]

    @property
    def metrics(self):
        return [self.acc_metric]

    def normalize_feat(self, f):
        return tf.math.l2_normalize(f, axis=1)

    def train_step(self, data):

        (x, t_probs), y = data

        step = tf.cast(self.optimizer.iterations, tf.float32)
        beta = tf.minimum(self.beta_max, step/5000.0*self.beta_max)

        with tf.GradientTape() as tape:

            s_logits = self.student(x, training=True)

            s_loss = self.ce(y, s_logits)

            kd_loss = self.kld(
                tf.nn.softmax(t_probs/self.T),
                tf.nn.softmax(s_logits/self.T)
            ) * (self.T*self.T)

            feat_loss = 0.0

            for i,(tm,sm) in enumerate(zip(self.t_feats,self.s_feats)):

                t_feat = gap(tf.stop_gradient(tm(x, training=False)))
                s_feat = gap(sm(x, training=True))

                s_feat = self.proj_heads[i](s_feat)

                t_feat = self.normalize_feat(t_feat)
                s_feat = self.normalize_feat(s_feat)

                feat_loss += self.mse(t_feat, s_feat)

            loss = self.alpha*s_loss + (1-self.alpha)*kd_loss + beta*feat_loss

        train_vars = self.student.trainable_variables

        for ph in self.proj_heads:
            train_vars += ph.trainable_variables

        grads = tape.gradient(loss, train_vars)
        self.optimizer.apply_gradients(zip(grads, train_vars))

        self.acc_metric.update_state(y, s_logits)

        return {"loss":loss, "accuracy":self.acc_metric.result(), "beta":beta}

    def test_step(self, data):

        (x, t_probs), y = data

        s_logits = self.student(x, training=False)
        self.acc_metric.update_state(y, s_logits)

        return {"accuracy":self.acc_metric.result()}

# ---------------------------------------------------------
# CREATE RUN2 SAVE DIRECTORY
# ---------------------------------------------------------
run2_dir = "/content/drive/MyDrive/Dense_StableFeatureKD_Run2"
os.makedirs(run2_dir, exist_ok=True)

run2_best_path  = os.path.join(run2_dir,"best_student_run2.keras")
run2_final_path = os.path.join(run2_dir,"final_student_run2.keras")

# ---------------------------------------------------------
# CALLBACK — SAVE BEST STUDENT
# ---------------------------------------------------------
class SaveBestStudent(tf.keras.callbacks.Callback):

    def __init__(self,path):
        super().__init__()
        self.best=-1.0
        self.path=path

    def on_epoch_end(self,epoch,logs=None):

        val_acc = logs.get("val_accuracy")

        if val_acc is None:
            print("⚠️ val_accuracy missing — skipping")
            return

        if val_acc > self.best:
            self.best = val_acc
            print(f"🔥 RUN2 Saving BEST STUDENT (val_accuracy={val_acc:.4f})")
            self.model.student.save(self.path)

# ---------------------------------------------------------
# BUILD DISTILLER + CONTINUE TRAINING
# ---------------------------------------------------------
distiller = StableFeatureDistiller(
    student, teacher,
    teacher_feat_models,
    student_feat_models
)

distiller.compile(optimizer=Adam(1e-4))

print("\n🚀 RUN2 Training for another 25 epochs...")

distiller.fit(
    train_seq,
    validation_data=val_seq,
    epochs=25,
    callbacks=[SaveBestStudent(run2_best_path)]
)

# FINAL SAVE RUN2
distiller.student.save(run2_final_path)

print("\n✅ RUN2 COMPLETE")
print("Best Run2 model saved at:", run2_best_path)
print("Final Run2 model saved at:", run2_final_path)



🚀 Starting RUN 2 — Loading BEST STUDENT from Run1
🔹 Generating teacher soft labels...
1262/1262 ━━━━━━━━━━━━━━━━━━━━ 114s 90ms/step
✅ Run1 student loaded successfully


✅ Teacher reloaded

🚀 RUN2 Training for another 25 epochs...
Epoch 1/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step - accuracy: 0.9693 - beta: 0.0252 - loss: 0.0630🔥 RUN2 Saving BEST STUDENT (val_accuracy=0.9588)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 344s 283ms/step - accuracy: 0.9693 - beta: 0.0252 - loss: 0.0629 - val_accuracy: 0.9588
Epoch 2/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 245s 243ms/step - accuracy: 0.9836 - beta: 0.0757 - loss: 0.0278 - val_accuracy: 0.9520
Epoch 3/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 237s 234ms/step - accuracy: 0.9887 - beta: 0.1262 - loss: 0.0211 - val_accuracy: 0.9532
Epoch 4/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.9911 - beta: 0.1767 - loss: 0.0185🔥 RUN2 Saving BEST STUDENT (val_accuracy=0.9751)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 236s 234ms/step - accuracy: 0.9911 - beta: 0.1767 - loss: 0.0185 - val_accuracy: 0.9751
Epoch 5/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 237s 234ms/step - accuracy: 0.9913 - beta: 0.2272 - loss: 0.0171 - val_accuracy: 0.9468
Epoch

In [ ]:
# =========================================================
# RUN 3 — CONTINUE TRAINING (RESNET101V2 STABILIZED FEATURE KD)
# =========================================================

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence, MeanSquaredError
from tensorflow.keras.optimizers import Adam
import os

print("\n🚀 Starting RUN 3 — Loading BEST STUDENT from Run2")

# ---------------------------------------------------------
# LOAD STUDENT FROM RUN1
# ---------------------------------------------------------
run2_best_student_path = "/content/drive/MyDrive/Dense_StableFeatureKD_Run2/final_student_run2.keras"

student = load_model(run2_best_student_path)

print("✅ Run1 student loaded successfully")

# ---------------------------------------------------------
# LOAD TEACHER AGAIN (SAFE PRACTICE)
# ---------------------------------------------------------
teacher_path = "/content/drive/MyDrive/Alzheimer_Models/DenseNet121_best_model.h5"

teacher = load_model(teacher_path)
teacher.trainable = False

print("✅ Teacher reloaded")

# -------------------------------
# DENSENET121 FEATURE MODELS
# -------------------------------

# 🔥 Correct DenseNet121 feature layers
teacher_feat_layers = [
    "pool4_conv",   # mid-level dense block output
    "relu"          # final activation before classifier
]

student_feat_layers = [
    "separable_conv2d_3",
    "separable_conv2d_6"
]

teacher_feat_models = [
    Model(teacher.input, teacher.get_layer(n).output)
    for n in teacher_feat_layers
]

student_feat_models = [
    Model(student.input, student.get_layer(n).output)
    for n in student_feat_layers
]

gap = GlobalAveragePooling2D()

# ---------------------------------------------------------
# STABILIZED DISTILLER (IDENTICAL LOGIC)
# ---------------------------------------------------------
class StableFeatureDistiller(tf.keras.Model):

    def __init__(self, student, teacher, t_feats, s_feats,
                 alpha=0.5, T=3.0, beta_max=0.25):
        super().__init__()

        self.student = student
        self.teacher = teacher
        self.t_feats = t_feats
        self.s_feats = s_feats
        self.alpha = alpha
        self.T = T
        self.beta_max = beta_max

        self.ce = CategoricalCrossentropy(from_logits=False)
        self.kld = KLDivergence()
        self.mse = MeanSquaredError()

        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')

        self.proj_heads = [
            Dense(tm.output_shape[-1], use_bias=False, name=f"proj_head_{i}")
            for i, tm in enumerate(self.t_feats)
        ]

    @property
    def metrics(self):
        return [self.acc_metric]

    def normalize_feat(self, f):
        return tf.math.l2_normalize(f, axis=1)

    def train_step(self, data):

        (x, t_probs), y = data

        step = tf.cast(self.optimizer.iterations, tf.float32)
        beta = tf.minimum(self.beta_max, step/5000.0*self.beta_max)

        with tf.GradientTape() as tape:

            s_logits = self.student(x, training=True)

            s_loss = self.ce(y, s_logits)

            kd_loss = self.kld(
                tf.nn.softmax(t_probs/self.T),
                tf.nn.softmax(s_logits/self.T)
            ) * (self.T*self.T)

            feat_loss = 0.0

            for i,(tm,sm) in enumerate(zip(self.t_feats,self.s_feats)):

                t_feat = gap(tf.stop_gradient(tm(x, training=False)))
                s_feat = gap(sm(x, training=True))

                s_feat = self.proj_heads[i](s_feat)

                t_feat = self.normalize_feat(t_feat)
                s_feat = self.normalize_feat(s_feat)

                feat_loss += self.mse(t_feat, s_feat)

            loss = self.alpha*s_loss + (1-self.alpha)*kd_loss + beta*feat_loss

        train_vars = self.student.trainable_variables

        for ph in self.proj_heads:
            train_vars += ph.trainable_variables

        grads = tape.gradient(loss, train_vars)
        self.optimizer.apply_gradients(zip(grads, train_vars))

        self.acc_metric.update_state(y, s_logits)

        return {"loss":loss, "accuracy":self.acc_metric.result(), "beta":beta}

    def test_step(self, data):

        (x, t_probs), y = data

        s_logits = self.student(x, training=False)
        self.acc_metric.update_state(y, s_logits)

        return {"accuracy":self.acc_metric.result()}

# ---------------------------------------------------------
# CREATE RUN2 SAVE DIRECTORY
# ---------------------------------------------------------
run3_dir = "/content/drive/MyDrive/Dense_StableFeatureKD_Run3"
os.makedirs(run3_dir, exist_ok=True)

run3_best_path  = os.path.join(run3_dir,"best_student_run3.keras")
run3_final_path = os.path.join(run3_dir,"final_student_run3.keras")

# ---------------------------------------------------------
# CALLBACK — SAVE BEST STUDENT
# ---------------------------------------------------------
class SaveBestStudent(tf.keras.callbacks.Callback):

    def __init__(self,path):
        super().__init__()
        self.best=-1.0
        self.path=path

    def on_epoch_end(self,epoch,logs=None):

        val_acc = logs.get("val_accuracy")

        if val_acc is None:
            print("⚠️ val_accuracy missing — skipping")
            return

        if val_acc > self.best:
            self.best = val_acc
            print(f"🔥 RUN3 Saving BEST STUDENT (val_accuracy={val_acc:.4f})")
            self.model.student.save(self.path)

# ---------------------------------------------------------
# BUILD DISTILLER + CONTINUE TRAINING
# ---------------------------------------------------------
distiller = StableFeatureDistiller(
    student, teacher,
    teacher_feat_models,
    student_feat_models
)

distiller.compile(optimizer=Adam(1e-4))

print("\n🚀 RUN3 Training for another 25 epochs...")

distiller.fit(
    train_seq,
    validation_data=val_seq,
    epochs=25,
    callbacks=[SaveBestStudent(run3_best_path)]
)

# FINAL SAVE RUN3
distiller.student.save(run3_final_path)

print("\n✅ RUN3 COMPLETE")
print("Best Run3 model saved at:", run3_best_path)
print("Final Run3 model saved at:", run3_final_path)



🚀 Starting RUN 2 — Loading BEST STUDENT from Run1
✅ Run1 student loaded successfully


✅ Teacher reloaded

🚀 RUN2 Training for another 25 epochs...
Epoch 1/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.9951 - beta: 0.0252 - loss: 0.0086🔥 RUN2 Saving BEST STUDENT (val_accuracy=0.9489)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 311s 264ms/step - accuracy: 0.9951 - beta: 0.0252 - loss: 0.0086 - val_accuracy: 0.9489
Epoch 2/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step - accuracy: 0.9964 - beta: 0.0757 - loss: 0.0084🔥 RUN2 Saving BEST STUDENT (val_accuracy=0.9586)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 236s 233ms/step - accuracy: 0.9964 - beta: 0.0757 - loss: 0.0083 - val_accuracy: 0.9586
Epoch 3/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 222ms/step - accuracy: 0.9951 - beta: 0.1262 - loss: 0.0087🔥 RUN2 Saving BEST STUDENT (val_accuracy=0.9617)
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 237s 235ms/step - accuracy: 0.9951 - beta: 0.1262 - loss: 0.0087 - val_accuracy: 0.9617
Epoch 4/25
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 236s 233ms/step - accuracy: 0.9935 - beta: 0.1767 - loss: 0.0085 - val_accuracy: 